# IMDB Movie Review Sentiment Analysis using RNNs

In this notebook we train a recurrent neural network (an LSTM) to decide whether a movie review is positive or negative. The same data pipeline is used in the next notebook (12.6), where we replace the LSTM with a transformer, so the two models can be compared directly.

## Step 1: Import Necessary Libraries and Setting Up the Environment

In this section, we import all the necessary libraries required for our sentiment classification task using Recurrent Neural Networks (RNNs) in PyTorch. We start by importing `torch` and its submodules like `nn` for neural network layers, `optim` for optimization algorithms, and the `DataLoader` and `Dataset` classes from `torch.utils.data`. From `torch.nn.utils.rnn` we import `pad_sequence`, which pads sequences to a uniform length, and `pack_padded_sequence`, which tells the LSTM how long each sequence really is so that it can skip the padding.

The data comes from the Hugging Face `datasets` library (`load_dataset`). We build the tokenizer and the vocabulary ourselves with Python's `re` (regular expressions) and `collections` modules, so that every step of the text pipeline is visible.

We also import `numpy`, a fundamental package for scientific computing, and `time` and `copy` to time the training and keep a copy of the best model.

Finally, we set the `device` variable to use a GPU if available (`cuda`), otherwise fallback to using the CPU. This ensures that our model training and inference processes are as efficient as possible, leveraging hardware acceleration when possible.

In [1]:
# Run this if you're in Colab (the datasets library is usually preinstalled there)
# %pip install datasets

In [2]:
import re
import collections
import copy
import time

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from datasets import load_dataset
import numpy as np

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## Step 2: Prepare the Data

### Setting the Seed for Reproducibility
To make the experiment repeatable, we set a random seed for PyTorch and NumPy. This controls the randomness in weight initialization, dropout, data shuffling, and the train/validation split. We also ask CuDNN to use deterministic algorithms.

### Loading the IMDB Dataset
We load the IMDB movie review dataset from the Hugging Face hub with `load_dataset("imdb")`. It contains 25,000 training reviews and 25,000 test reviews, each labeled 0 (negative) or 1 (positive), with the two classes balanced (it also contains 50,000 unlabeled reviews that we do not use). The first call downloads the data (about 80 MB), so it needs internet access.

### Training, Validation, and Test Sets
We hold out 10% of the training split (2,500 reviews, chosen with a fixed seed) as a **validation set**. We use it to monitor the model after every epoch and to pick the best epoch. The **test set** is used only once, at the very end, to report the final accuracy. If we used the test set to monitor training and choose the epoch, the reported test accuracy would be optimistically biased.

### Tokenization and Vocabulary Building
A tokenizer splits each review into a list of tokens. We use a simple regular expression: after lowercasing the text and removing the HTML line breaks (`<br />`) that appear in many reviews, every run of letters, digits, and apostrophes becomes a word token and every other non-space character (such as punctuation) becomes a token of its own. For example, `"Don't watch it!"` becomes `["don't", "watch", "it", "!"]`.

We then build a **vocabulary** that maps each token to an integer index. The vocabulary is built from the training reviews only (never from validation or test data), and keeps the `MAX_VOCAB` = 20,000 most frequent tokens that occur at least `MIN_FREQ` = 5 times. Two special tokens are added: `<pad>` (index 0), used to fill short sequences, and `<unk>` (index 1), which replaces every token that is not in the vocabulary.

### Data Processing Functions
The `encode` function converts a raw review into a tensor of token indices and truncates it to at most `MAXLEN` tokens. We encode every review once, before training. The `collate_batch` function combines a list of encoded reviews into a batch: it pads them with `<pad>` to the length of the longest review in the batch and also returns the true (unpadded) length of each review, which the model needs in order to ignore the padding.

### DataLoader Setup
Finally, we wrap the encoded reviews in a small PyTorch `Dataset` class and create a `DataLoader` for each of the three sets. Only the training loader shuffles the data.

In [3]:
# Hyperparameters
BATCH_SIZE = 64
MAXLEN = 500      # Maximum length of a review (in tokens); longer reviews are truncated
MAX_VOCAB = 20000 # Keep at most this many of the most frequent tokens
MIN_FREQ = 5      # Ignore tokens that appear fewer than this many times in the training data

# Set seed for reproducibility
SEED = 1234
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True

# Load the IMDB dataset (labels: 0 = negative, 1 = positive)
ds = load_dataset("imdb")

# Hold out 10% of the training split as a validation set
split = ds["train"].train_test_split(test_size=0.1, seed=SEED)
train_data, valid_data, test_data = split["train"], split["test"], ds["test"]
print(f"Training reviews: {len(train_data)}, validation reviews: {len(valid_data)}, test reviews: {len(test_data)}")

# Tokenizer: lowercase, remove HTML line breaks, split into words and punctuation
def tokenize(text):
    return re.findall(r"[a-z0-9']+|[^\sa-z0-9']", text.lower().replace("<br />", " "))

print(tokenize("Don't watch it!<br />It was 2 hours of my life..."))

# Build the vocabulary from the training reviews only
counter = collections.Counter()
for text in train_data["text"]:
    counter.update(tokenize(text))
PAD_IDX, UNK_IDX = 0, 1
itos = ["<pad>", "<unk>"] + [tok for tok, freq in counter.most_common(MAX_VOCAB) if freq >= MIN_FREQ]
stoi = {tok: idx for idx, tok in enumerate(itos)}
vocab_size = len(itos)
print(f"Distinct tokens in the training reviews: {len(counter)}, vocabulary size: {vocab_size}")


# Function to convert a review into a tensor of at most MAXLEN token indices
def encode(text):
    ids = [stoi.get(tok, UNK_IDX) for tok in tokenize(text)]
    return torch.tensor(ids[:MAXLEN], dtype=torch.int64)


# Encode every review once and keep the results in memory
class IMDBDataset(Dataset):
    def __init__(self, hf_data):
        self.texts = [encode(text) for text in hf_data["text"]]
        self.labels = hf_data["label"]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.labels[idx], self.texts[idx]


# Function to collate data samples into batch tensors
def collate_batch(batch):
    labels = torch.tensor([label for label, _ in batch], dtype=torch.int64)
    text_list = [text for _, text in batch]
    lengths = torch.tensor([text.size(0) for text in text_list], dtype=torch.int64)
    texts = pad_sequence(text_list, batch_first=True, padding_value=PAD_IDX)
    return labels, texts, lengths


train_dataset = IMDBDataset(train_data)
valid_dataset = IMDBDataset(valid_data)
test_dataset = IMDBDataset(test_data)

all_lengths = np.array([len(tokenize(text)) for text in train_data["text"]])
print(f"Median review length: {np.median(all_lengths):.0f} tokens; {np.mean(all_lengths > MAXLEN)*100:.1f}% of the reviews are truncated to {MAXLEN} tokens")

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
valid_dataloader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)

Training reviews: 22500, validation reviews: 2500, test reviews: 25000
["don't", 'watch', 'it', '!', 'it', 'was', '2', 'hours', 'of', 'my', 'life', '.', '.', '.']


Distinct tokens in the training reviews: 83341, vocabulary size: 20002


Median review length: 201 tokens; 11.3% of the reviews are truncated to 500 tokens


## Step 3: Define the RNN Model

We define a simple RNN model for sentiment classification. This model consists of an embedding layer, an LSTM layer, and a linear layer. The embedding layer converts integer indices into dense vectors, which are then processed by the LSTM layer. The LSTM reads the sequence one token at a time and updates its hidden state $\mathbf{h}_t$ after each token. After the whole review has been read, the final hidden state summarizes the review, and a linear layer maps it to a single number (a logit) whose sign gives the predicted sentiment.

### Padding and packed sequences
The reviews in a batch have different lengths, so the shorter ones are padded with `<pad>` tokens at the end. We must be careful about which hidden state we give to the classifier. If we simply took the LSTM output at the last position of the padded batch (`x[:, -1, :]`), then for every review shorter than the longest one in the batch we would be using the state *after the LSTM has read many padding tokens*, not the state at the end of the actual review. The information from the review would have to survive a long run of meaningless pad inputs, and the model then struggles to learn at all (an earlier version of this notebook made this mistake and stayed close to 50% accuracy for many epochs).

Instead, we pass the true lengths to `nn.utils.rnn.pack_padded_sequence`. The packed sequence tells the LSTM where each review ends, so it stops updating each review's state at its last real token, and the final hidden state `hn` returned by the LSTM is the state at the end of each review. We classify from `hn[-1]` (the last layer's final hidden state). The `padding_idx` argument of the embedding layer keeps the embedding of `<pad>` fixed at zero.

The model outputs a logit rather than a probability: the sigmoid is folded into the loss function (next step), which is numerically more stable.

In [4]:
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, lstm_units, pad_idx=PAD_IDX):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.dropout = nn.Dropout(0.2)
        self.lstm = nn.LSTM(embed_dim, lstm_units, batch_first=True)
        self.linear = nn.Linear(lstm_units, 1)

    def forward(self, x, lengths):
        x = self.emb(x)
        x = self.dropout(x)
        # Pack the padded batch so that the LSTM stops at the true end of each review
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (hn, cn) = self.lstm(packed)  # hn has shape (num_layers, batch, lstm_units)
        x = self.dropout(hn[-1])         # final hidden state of each review
        return self.linear(x).view(-1)   # one logit per review

## Step 4: Instantiate the Model, Loss, and Optimizer

In this section, we initialize the RNN model with specific hyperparameters and prepare it for training. This involves defining the model architecture, setting up the optimizer, and specifying the loss function.

### Model Configuration
- `VOCAB_SIZE` is the total number of unique tokens in our vocabulary (including the two special tokens).
- `EMBED_DIM` specifies the dimensionality of the word embeddings.
- `LSTM_UNITS` is the number of units in the LSTM layer (the size of the hidden state).

### Model Initialization
The `SimpleRNN` is initialized with the specified hyperparameters to create an instance of our sentiment classification model. We also count its trainable parameters. Most of them are in the embedding table (one vector of size `EMBED_DIM` for every token in the vocabulary).

### Optimizer and Loss Function
- The Adam optimizer (learning rate $10^{-3}$) is chosen for its efficiency and effectiveness in handling sparse gradients and adapting the learning rate for each parameter.
- The binary cross-entropy loss is used as the criterion for training, which is the right loss for a binary classification task (see chapter 5 of the book). We use `nn.BCEWithLogitsLoss`, which applies the sigmoid to the model's logit and computes the binary cross-entropy in a single, numerically stable step. This is why the model has no sigmoid layer; we apply `torch.sigmoid` ourselves only when we turn the logits into predicted labels.

In [5]:
VOCAB_SIZE = vocab_size
EMBED_DIM = 128   # Size of each word embedding
LSTM_UNITS = 128  # Number of hidden units in the LSTM

model = SimpleRNN(VOCAB_SIZE, EMBED_DIM, LSTM_UNITS).to(device)
print(model)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()

SimpleRNN(
  (emb): Embedding(20002, 128, padding_idx=0)
  (dropout): Dropout(p=0.2, inplace=False)
  (lstm): LSTM(128, 128, batch_first=True)
  (linear): Linear(in_features=128, out_features=1, bias=True)
)
Trainable parameters: 2,692,481


## Step 5: Training and Validation Loops

This section outlines the core of our model training and evaluation process. It includes two distinct parts: the training loop and the validation loop.

- **Training Loop**: The training loop is where the model learns from the training dataset. It involves setting the model to training mode, iterating over the training data, computing the loss (using the specified criterion), and updating the model parameters based on the computed gradients. We clip the norm of the gradient to 1.0, a common safeguard against the occasional very large gradients of recurrent networks. This loop also calculates and accumulates the training loss and accuracy to monitor the performance of the model during training.

- **Validation Loop**: The validation loop is executed after the training loop and is used to evaluate the model's performance on the held-out validation set, which it does not see during training. The model is set to evaluation mode, which disables certain layers/functions like dropout, specific to training. Similar to the training loop, it computes the loss and accuracy but does not update the model's parameters, and it runs inside `torch.no_grad()` because no gradients are needed.

- **Epoch Iteration**: Both loops are encapsulated within an epoch iteration loop, where an epoch represents a single pass through the entire training dataset. The number of epochs is a hyperparameter that defines the number of times the learning algorithm will work through the entire training dataset. After each epoch, the model's performance is assessed using both the training and validation data, and we keep a copy of the weights from the epoch with the best validation accuracy.

In both loops, the prediction is positive when the predicted probability `torch.sigmoid(logits)` is greater than 0.5 (equivalently, when the logit is greater than zero).

In [6]:
# Training loop
def train(model, dataloader, optimizer, criterion, device):
    model.train()  # Set the model to training mode
    epoch_loss = 0
    correct = 0
    total = 0
    for labels, texts, lengths in dataloader:
        labels = labels.to(device)
        texts = texts.to(device)

        optimizer.zero_grad()  # Clear the gradients
        logits = model(texts, lengths)  # Forward pass: compute the output
        loss = criterion(logits, labels.float())  # Compute the loss
        loss.backward()  # Backward pass: compute the gradient of the loss with respect to model parameters
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Clip the gradient norm
        optimizer.step()  # Update the model parameters

        epoch_loss += loss.item()  # Accumulate the loss

        threshold = 0.5  # Probability threshold for the predictions. If the predicted probability
                         # is greater than this threshold, we predict class 1, else we predict class 0.
        y_pred = (torch.sigmoid(logits) > threshold).int()  # Convert the probabilities to binary predictions
        correct += (y_pred == labels).type(torch.float).sum().item() # Count the number of correct predictions
        total += labels.size(0)

    accuracy = correct / total  # Compute the accuracy
    return epoch_loss / len(dataloader), accuracy  # Return the average loss and accuracy


# Validation loop
def evaluate(model, dataloader, criterion, device):
    model.eval()  # Set the model to evaluation mode
    epoch_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():  # No gradients needed
        for labels, texts, lengths in dataloader:
            labels = labels.to(device)
            texts = texts.to(device)

            logits = model(texts, lengths)  # Forward pass
            loss = criterion(logits, labels.float())  # Compute loss

            epoch_loss += loss.item()  # Accumulate the loss

            threshold = 0.5  # Probability threshold for the predictions. If the predicted probability
                             # is greater than this threshold, we predict class 1, else we predict class 0.
            y_pred = (torch.sigmoid(logits) > threshold).int()  # Convert the probabilities to binary predictions
            correct += (y_pred == labels).type(torch.float).sum().item() # Count the number of correct predictions
            total += labels.size(0)

    accuracy = correct / total  # Compute the accuracy
    return epoch_loss / len(dataloader), accuracy  # Return the average loss and accuracy


N_EPOCHS = 5
best_valid_accuracy = 0.0
start_time = time.time()
for epoch in range(N_EPOCHS):
    train_loss, train_accuracy = train(model, train_dataloader, optimizer, criterion, device)
    valid_loss, valid_accuracy = evaluate(model, valid_dataloader, criterion, device)
    if valid_accuracy > best_valid_accuracy:
        best_valid_accuracy = valid_accuracy
        best_epoch = epoch + 1
        best_state = copy.deepcopy(model.state_dict())  # Keep the best weights in memory
    print(f'Epoch: {epoch+1}, Train Loss: {train_loss:.3f}, Train Acc: {train_accuracy*100:.2f}%, Validation Loss: {valid_loss:.3f}, Validation Acc: {valid_accuracy*100:.2f}%')
train_time = time.time() - start_time
print(f"Training time: {train_time:.1f} s")
print(f"Best validation accuracy: {best_valid_accuracy*100:.2f}% (epoch {best_epoch})")

Epoch: 1, Train Loss: 0.627, Train Acc: 63.78%, Validation Loss: 0.559, Validation Acc: 72.60%


Epoch: 2, Train Loss: 0.495, Train Acc: 77.29%, Validation Loss: 0.461, Validation Acc: 79.92%


Epoch: 3, Train Loss: 0.398, Train Acc: 83.02%, Validation Loss: 0.426, Validation Acc: 82.32%


Epoch: 4, Train Loss: 0.332, Train Acc: 86.14%, Validation Loss: 0.403, Validation Acc: 85.48%


Epoch: 5, Train Loss: 0.278, Train Acc: 88.90%, Validation Loss: 0.403, Validation Acc: 85.36%
Training time: 72.2 s
Best validation accuracy: 85.48% (epoch 4)


## Step 6: Evaluate on the Test Set

Now that training is finished, we restore the weights from the epoch with the best validation accuracy and evaluate the model on the test set. This is the only time the test set is used, so the test accuracy is an unbiased estimate of how well the model works on new reviews.

TODO: Before you run the cell, predict the test accuracy from the validation accuracies above. Should it be higher, lower, or about the same? Why?

In [7]:
model.load_state_dict(best_state)
test_loss, test_accuracy = evaluate(model, test_dataloader, criterion, device)
print(f'Test Loss: {test_loss:.3f}, Test Acc: {test_accuracy*100:.2f}%')

print("\nSummary")
print(f"Trainable parameters: {n_params:,}")
print(f"Epochs: {N_EPOCHS} (best validation epoch: {best_epoch})")
print(f"Best validation accuracy: {best_valid_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")
print(f"Training time: {train_time:.1f} s on {device}")

Test Loss: 0.407, Test Acc: 84.72%

Summary
Trainable parameters: 2,692,481
Epochs: 5 (best validation epoch: 4)
Best validation accuracy: 85.48%
Test accuracy: 84.72%
Training time: 72.2 s on cuda


## Step 7: Try Your Own Reviews

The function below encodes a new review with the same tokenizer and vocabulary and prints the predicted probability that it is positive.

TODO: Write a few reviews of your own, including some tricky ones (for example with negation, such as "not bad at all", or sarcasm). Where does the model fail?

In [8]:
def predict_sentiment(model, text):
    model.eval()
    ids = encode(text).unsqueeze(0).to(device)
    lengths = torch.tensor([ids.size(1)])
    with torch.no_grad():
        prob = torch.sigmoid(model(ids, lengths)).item()
    return prob

for review in ["This movie was fantastic! The acting was great and the story kept me hooked.",
               "A boring, predictable film. I wanted my two hours back.",
               "Not bad at all, I actually enjoyed it."]:
    print(f"{predict_sentiment(model, review):.3f}  {review}")

0.960  This movie was fantastic! The acting was great and the story kept me hooked.
0.041  A boring, predictable film. I wanted my two hours back.
0.329  Not bad at all, I actually enjoyed it.
